### Klassen
Nachstehende eine typische Klassendefinition (Teil unserer Vector-Klasse).    
- Alle im Klassenbody einer Klasse `Vector` definierten Attribute (Variabeln und Funktionen) lassen sich von ausserhalb der Klasse mit `Vector.<name>` ansprechen.
- Ist `v` eine Instanz der Klasse `Vector` und wird versucht, ein Attribut zu lesen, welches die Instanz nicht hat, wird das entsprechende Attribut der Klasse geliefert.
- Ist `f` eine in `Vector` definierte Funktion (aber kein Attribut von `v`)
  wird `v.f(..)` zu `Vector.f(v, ...)`.
  Das ermöglicht auf die aufrufenden Instanz `v` und alle ihre Attribute und Methoden zuzugreifen.

- Die Decorators `staticmethod` und `classmethod` ändern, woran
das erste Argument der dekorierten Methode gebunden wird.
- Methoden wie `__add__(self, other)` heissen Magic- oder dunder-Methoden und
  werden aufgerufen, wenn die zugehörige Operation auf Instanzen der Klasse angesendet wird.

  

```python
class Vector:
    PI = 3.1415926535  # Klassenattribut
    
    def __init__(self, x, y):
        self.x = x  # Attribute der Instanz
        self.y = y

    # Vector.deg2rad(degree) führt deg2rad(degree) aus
    # v.deg2rad(degree)      führt deg2rad(degree) aus
    @staticmethod
    def deg2rad(degree):
        return degree * self.PI / 180  # beachte self.PI (PI waere globale Variable)
        
    # Vector.from_pts(q, p) führt from_pts(Vector, p, q)  aus
    # v.from_pts(q, p)      führt from_pts(type(v), p, q) aus
    @classmethod  
    def from_pts(cls, p, q):  
        '''gibt den Vector q-p zurueck'''
        return cls(*q) - cls(*p)

    # Vector.norm(v) führt norm(v) aus
    # v.norm()       führt norm(v) aus
    def norm(self):
        '''gib die Laenge von v zurueck'''
        return (self.x**2 + self.y**2)**.5

    # v + w triggert __add__(v, w)
    def __add__(self, other):
        '''gibt den Vektor self+other zurueck'''
        return Vector(self.x+other.x, self.y+other.y)

    # wird aufgerufen, falls Instanz als String dargestellt werden soll
    def __repr__(self):
        return f'Vec({self.x:.2f}, {self.y:.2f})'
```

***
### Getter und Setter
Oft möchte man vor dem
Setzen eines Attribute prüfen, ob der Wert sinnvoll ist, oder den Wert noch
in eine einheitliche Form bringen. 
Manchmal erfordert das Setzen eines Attributes auch das Modifizieren eines anderen Attributes.

In solchen Fällen kann man ein Attribut `foo` zu einem **privaten** Attribut `_foo` machen,
und zum **Lesen** und **Setzen Methoden** `get_foo(self)` und `set_foo(self, value)` **bereitstellen** (getter und setter).  

Der Benutzer sollte das Attribut dann nur mit dem Getter (Setter) lesen (setzen).

In [ ]:
class UserProfile:
    '''Es wird geprueft, ob der username ein str ist
       vor dem Zuweisen wird Whitespace entfernt und in lowercase umgewandelt
    '''
    def __init__(self, username):
        self.set_username(username)

    def get_username(self):
        return self._username  # unsername wird als privates Attribut gespeichert

    def set_username(self, value):
        if not isinstance(value, str):
            raise TypeError('username must be a str!')
        self._username = value.strip().lower()

    def __repr__(self):
        return f'UserProfile({self._username})'

In [ ]:
userprofile = UserProfile('Mike_123  ')
print(f'username: {userprofile.get_username()}')
userprofile

In [ ]:
class Polygon:
    '''Wird der anchor neu gesetzt, sollen alle pts
       die Verschiebung des anchors mitmachen
    '''
    def __init__(self, pts, anchor):
        self.pts = pts
        self._anchor = anchor

    def get_anchor(self):
        return self._anchor

    def set_anchor(self, new_anchor):
        dx, dy = new_anchor[0] - self._anchor[0], new_anchor[1] - self._anchor[1]
        self.pts = [(x+dx, y+dy) for x, y in self.pts]
        self._anchor = new_anchor

    def __repr__(self):
        return f'Polygon(pts={self.pts}, anchor={self._anchor})'

In [ ]:
A, B, C = (-1, 1), (0, -1), (1, 1)
ORIGIN = (0, 0)
poly = Polygon(pts=[A, B, C], anchor=ORIGIN)
poly.set_anchor((10, 10))
poly

***
### Getter und Setter mit @property
Python erlaubt, die Verwendung von Getter und Setter zu verstecken, indem ein Attribute zu
einer Property gemacht wird. Auf folgende Weise wird das Attribut `foo` zu einer Property gemacht.

```python
class A:
    @property  # Getter Attribut foo
    def foo(self):
        return self._foo  
    
    @foo.setter  # Setter fuer Attribut foo (optional)
    def foo(self):
        # do stuff
        self._foo = foo
```

- Jeder Zugriff auf `a.foo` triggert den Aufruf des Getter 
- Jede Zuweisung  `a.foo = value` triggert den Aufruf des Setters

In [ ]:
class UserProfile:
    '''Es wird geprueft, ob der username ein str ist
       vor dem Zuweisen wird Whitespace entfernt und in lowercase umgewandelt
    '''
    def __init__(self, username):
        self.username = username  # benutzt setter fuer username

    @property  # getter fuer username
    def username(self):
        return self._username

    @username.setter  # setter fuer username
    def username(self, value):
        if not isinstance(value, str):
            raise TypeError('username must be a str!')
        self._username = value.strip().lower()

    def __repr__(self):
        return f'UserProfile({self.username})'

In [ ]:
userprofile = UserProfile('Mike_123  ')
print(f'username: {userprofile.username}')
userprofile

In [ ]:
class Polygon:
    '''Wird der anchor neu gesetzt, sollen alle pts
       die Verschiebung des anchors mitmachen
    '''
    def __init__(self, anchor, pts):
        self.pts = pts
        self._anchor = anchor  # setter soll hier nicht benutzt werden

    @property
    def anchor(self):
        return self._anchor

    @anchor.setter
    def anchor(self, new_anchor):
        dx, dy = new_anchor[0] - self.anchor[0], new_anchor[1] - self.anchor[1]
        self.pts = [(x+dx, y+dy) for x, y in self.pts]
        self._anchor = new_anchor

    def __repr__(self):
        return f'Polygon(pts={self.pts}, anchor={self.anchor})'

In [ ]:
A, B, C = (-1, 1), (0, -1), (1, 1)
ORIGIN = (0, 0)
poly = Polygon(pts=[A, B, C], anchor=ORIGIN)
poly.anchor = (10, 10)
poly

***
### Vererbung
Ist z.B. `Animal` eine Klasse, so kann eine neue Klasse von `Animal` erben, mit
```python
class Dog(Animal)
    ...
```

Die Klasse `Dog` hat dann alle Attribute (Variablen und Methoden von Animal).  
In der Klasse `Dog` kann man die Methoden von `Animal` überschreiben und
bei Bedarf die Methode der Elternklasse `Animal` nutzen.
```python
class Animal:
    def __init__(self, name):
        self.name = name
        self.sound = 'Grrr!'

    def talk(self):  # Methode wird ueberschrieben
        print(f'{self.sound}')
        super().talk()  # Methode der Elterklasse aufrufen

class Dog(Animal):
    def __init__(self, name, sound):
        super().__init__(name)  # ruft Animal(self, name) auf
        self.sound = sound
```

**Grösseres Beispiel**:  
Siehe `Model_View_Controller_Pattern/Einfache_MVC_Implementation.ipynb`

In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name
        self.sound = 'Grrr!'

    def talk(self):
        print(f'{self.sound}')

    def __repr__(self):
        classname = type(self).__name__
        return f'{classname}(name={self.name})'


class Dog(Animal):
    def __init__(self, name):
        super().__init__(name)  # ruft Animal.__init__(self, name) auf
        self.sound = 'wuff'


class Cat(Animal):
    def __init__(self, name):
        super().__init__(name)
        self.sound = 'miau'

    def talk(self):  # Methode ueberschreiben
        print(f'{3*self.sound[0]}{self.sound[1:]}')


class Frog(Animal):
    def __init__(self, name):
        super().__init__(name)
        self.sound = 'quak'

    def talk(self):  # Methode modifizieren
        print('listen:', end=' ')
        super().talk()

In [ ]:
dog = Dog('Fido')
cat = Cat('Mia')
frog = Frog('Quak')

dog, cat, frog

In [ ]:
dog.talk()
cat.talk()
frog.talk()

### Komposition
Siehe `Klassen_Komposition.ipynb`.